# Mod 11 時間序列 Time Series
> pandas 最基本的時間序列型別就是以時間戳（TimeStamp）為 `index` 元素的 Series 型別。

> pandas提供了一組標準的時間序列，輕鬆地`進行切片/切塊、聚合、對定期/不定期的時間序列進行重取樣`等。這些工具中大部分都對金融和經濟資料尤其有用。

> pandas中有四種時間型別：

> * Date times : 日期和時間，可以帶時區。和標準庫中的 datetime.datetime 類似。
> * Time deltas： 絕對持續時間，和 標準庫中的 datetime.timedelta 類似。
> * Time spans： 由時間點及其關聯的頻率定義的時間跨度。
> * Date offsets：基於日曆計算的時間 和 dateutil.relativedelta.relativedelta 類似。

> ### 摘要 ：

類型         | 說明            | 類似 Python 類別     | pandas 物件    
------------ | -------------- | -------------------- | --------------
Date times   | 單一時間點      | `datetime.datetime`  | `Timestamp`   
Time deltas  | 時間差          | `datetime.timedelta` | `Timedelta`   
Time spans   | 具有頻率的時間段 | 無直接對應           | `Period`    
Date offsets | 日曆偏移量      | `relativedelta`      | `DateOffset`, `MonthEnd`, `YearBegin`, 等 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 11-1 Date and Time Data Types and Tools

### Pandas NOW

In [ ]:
pd.to_datetime("today"), pd.Timestamp('now'), pd.Timestamp.now()   

### pd.to_datetime / Index

In [ ]:
datestrs = ['2021-07-06 12:00:00', '2018-08-06 00:00:00']
pd.to_datetime(datestrs)

### Timestamp 時間戳 : 時間戳表示某個具體的時間點

In [ ]:
pd.Timestamp(2012, 5, 1), pd.Timestamp('2012-05-01'), pd.to_datetime('2012-05-01')

### parse

In [ ]:
from dateutil.parser import parse
parse('2011-01-03'), parse('Jan 31, 1997 10:45 PM'), parse('6/12/2011', dayfirst=True)

## Timedelta 是 Pandas 庫中用來表示時間間隔（Duration）或時間差的物件

In [ ]:
import pandas as pd

# 1. 建立一個時間點 (Timestamp)
start_time = pd.Timestamp("2026-06-13 10:00:00")
print(f"開始時間\t\t: {start_time}")

# 2. 建立一個時間間隔 (Timedelta) -> 2天又5個小時
time_delay = pd.Timedelta(days=2, hours=5)
# 也可以用字串建立：pd.Timedelta('2 days 5 hours')
print(f"時間間隔\t\t: {time_delay}")

In [ ]:
# 3. 進行時間運算（時間點 + 時間間隔）
end_time = start_time + time_delay
print(f"結束時間\t\t: {end_time}\n"
      f"結束時間 date\t: {end_time.date()}, time : {end_time.time()}\n")

# 4. 計算兩個時間點的差值（會自動自動生成 Timedelta）
time_diff = end_time - start_time
print(f"兩者差距\t\t: {time_diff}\n"
      f"型態為\t\t: {type(time_diff)})")

## 11-2 Time Series Basics

In [ ]:
from datetime import datetime
dates = [datetime(2021, 1, 2), datetime(2021, 1, 5), datetime(2021, 1, 7),
         datetime(2021, 1, 8), datetime(2021, 1, 10), datetime(2021, 1, 12)]
ts = pd.Series(np.random.randn(6), index=dates)   # date index
print(f'{ts}\n\n'
      f'{ts.index}\n\n'
      f'{ts[::2]}\n\n'
      f'{ts+ts[::2]}')

### Indexing, Selection, Subsetting
#### pd.date_range

In [ ]:
pd.date_range('1-1-2018', periods=1000, freq='D')  # periods default freq='D'

In [ ]:
data = np.random.randint(1000, 5000, (1000))  # 1000 to 4999 pick 1000
series1 = pd.Series(data,  index = pd.date_range('1-1-2019', periods=1000, freq='B'))  # periods
print (f'series1 :\n{series1}\n\n')

series1.sort_values(ascending=True, inplace = True)  # note : inplace
# series1.sort_values(ascending=False, inplace = True)
# series1.sort_index(ascending=False, inplace = True)
print(f'series1.sort_values :\n{series1}\n\n'
      f'series1.index[:6] :\n{series1.index[:6]}\n\n'      # series1 已被排序
      f'series1.index.year[:6] :\n{series1.index.year[:6]}\n\n'
      f'series1.index.month[:6] :\n{series1.index.month[:6]}')

### 所有紀錄中, 最高紀錄前 5 筆 ? 最低紀錄 5 筆 ? 

In [ ]:
print(f'series1.sort_values :\n{series1.sort_values(ascending=False).iloc[0:5]}\n')

series1.sort_values(ascending=False, inplace= True)
print(f'series1.iloc[0:5] :\n{series1.iloc[0:5]}\n\n'
      f'series1.head() :\n{series1.head()}\n\n'
      f'series1.tail(5) :\n{series1.tail(5)}\n\n'
      f'series1.describe :\n{series1.describe()}')

### slice by year, month or trancate

In [ ]:
ts = pd.Series(np.random.randint(10, 100, 1000), index=pd.date_range('1/1/2018', periods=1000))
print(f'ts :\n{ts}\n\n'
      f"ts['2018'] :\n{ts['2018']}\n\n"
      f"ts['2018-05'][:] :\n{ts['2018-05'][:]}\n\n"
      f"ts.truncate(after='1/9/2018') :\n{ts.truncate(after='1/9/2018')}\n\n"
      f"ts.truncate(before='1/9/2018') :\n{ts.truncate(before='1/9/2018')}")

### resample 
### 常用頻率與日期偏移量

頻率                 |            日期偏移量 | 說明
--------------------:|---------------------|-----
D                     | Day                 | 日曆日
B	                  | BusinessDay         | 工作日
H                     | Hour	            | 小時
T 或 min              | Minute	            | 分
S                     | Second	            | 秒
L 或 ms	              | Milli	            | 毫秒
U	                  | Micro	            | 微秒
ME	                  | MonthEnd	        | 每月最後一個日曆日
BME	                  | BusinessMonthEnd    | 每月最後一個工作日
MS	                  | MonthBegin          | 每月第一個日曆日
BMS	                  | BussinessMonthBegin | 每月第一個工作日
W-MON、W-TUE、…	      | Week                | 指定星期幾 (MON、TUE、WED、THU、FRI、SAT、SUN)
WOM-1MON、WOM-2MON、… | WeekOfMonth	        | 產生每月第一、第二、第三或第四周的星期幾。例如WOM-3FRI表示每月第3個星期五
Q-JAN、Q-FEB、…	      | QuarterEnd	         | 以指定月份結束的年度，每季度最後一個月的最後一個日曆日
BQ-JAN、BQ-FEB、…	  | BusinessQuarterEnd	 | 以指定月份結束的年度，每季度最後一個月的最後一個工作日
QS-JAN、QS-FEB、…	  | QuarterBegin	     | 以指定月份結束的年度，每季度最後一個月的第一個日曆日
BQS-JAN、BQS-FEB、…	  | BusinessQuarterBegin | 以指定月份結束的年度，每季度最後一個月的第一個工作日
A-JAN、A-FEB、…	      | YearEnd              | 每年指定月份的最後一個日曆日
BA-JAN、BA-FEB、…	  | BusinessYearEnd	     | 每年指定月份的最後一個工作日
AS-JAN、AS-FEB、…	  | YearBegin	         | 每年指定月份的第一個日曆日
BAS-JAN、BAS-FEB、…	  | BusinessYearBegin	 | 每年指定月份的第一個工作日

In [ ]:
resampler = ts.resample(rule='BME').sum()  # try sum 2D : 2 days, M
print(f'{ts[:10]}\n\n'
      f'{resampler[:10]}\n')

### freq 設置一定的時間間隔
http://liao.cpython.org/pandas34.html

In [ ]:
dates = pd.date_range('1-1-2020', periods=100, freq='W-WED')
# dates = pd.date_range('1/1/2000', periods=100, freq='Y')
long_df = pd.DataFrame(np.random.randn(100, 4), index=dates,
                       columns=['Colorado', 'Texas', 'New York', 'Ohio'])
print(f'long_df :\n{long_df}\n\n\n'
      f"long_df.loc['12/2020'] :\n{long_df.loc['12/2020']}")

## 11-3 Date Ranges, Frequencies, and Shifting

### Generating Date Ranges

In [ ]:
index = pd.date_range('2021-04-01', '2021-06-01', freq='W')  # try freq = 'W', 'S'
index

### fix start or end point

In [ ]:
pd.date_range(start='2021-04-01', periods=20), pd.date_range(end='2021-06-01', periods=20)

### BM : business month end frequency

In [ ]:
pd.date_range('2000-01-01', '2000-12-01', freq='BME')  # BM : business month end frequency

In [ ]:
pd.date_range('2012-05-02 12:56:31', periods=5)  # base on days

In [ ]:
pd.date_range('2012-05-02 12:56:31', periods=5, normalize=True)  # date only

### Week of month dates

In [ ]:
rng = pd.date_range('2012-01-01', '2012-09-01', freq='WOM-3FRI')  # 3th Fridays of each month
rng

# `練習 Q02 (6,7,8)`

---

# Mod 12 Data Loading, Storage
><img src="./img/data_loading.jpg"  style='width:100%'>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 12-1 Reading and Writing Data in Text Format

## read csv

In [ ]:
df = pd.read_csv('./data/ex1.csv')
df, type(df), df.index, df.columns, df.values

In [ ]:
df

### observation

In [ ]:
print(f'df :\n{df}\n\n'
      f'df.index\t: {df.index}\n'
      f'df.columns\t: {df.columns}\n\n'
      f'df.values :\n{df.values}\n\n'
      f'type(df.values) :\n{type(df.values)}\n'
      f'df.ndim : {df.ndim}\n'
      f'df.shape : {df.shape}\n\n'
      f'df.describe() :\n{df.describe()}\n\n'
      f'df.info :\n{df.info()}\n\n'
      f'df.head(4) :\n{df.head(4)}\n\n'          # default top 5 rows
      f'df.tail(4) :\n{df.tail(4)}')

### read table

In [ ]:
pd.read_table('data/ex1.csv', sep=',')  # try sep=','
# pd.read_table('data/ex1.csv', sep=',')

### read  with name : ex2.csv no header/ column name

In [ ]:
# pd.read_csv('data/ex2.csv')
# pd.read_csv('data/ex2.csv', header=None)  # try header = 0, 1
pd.read_csv('data/ex2.csv', names=['aa', 'bb', 'cc', 'dd', 'message'])  # try to remove 'b' or ...

### read  with name / index_col

In [ ]:
names = ['aa', 'bb', 'cc', 'dd', 'message']
pd.read_csv('data/ex2.csv', names=names, index_col='message')

## read : txt

In [ ]:
# open('data/ex3.txt')
list(open('./data/ex3.txt'))

In [ ]:
result = pd.read_table('data/ex3.txt', sep=r'\s+')   # re \s+ space
# result = pd.read_table('data/ex3.txt', sep=' ')   # re \s+ space
result

## 記憶體內的文字資料串流也可以使用 StringIO 物件建立, StringIO 是Python 的一個模組，它允許你`像操作檔案一樣操作字串`

In [ ]:
from io import StringIO
data = """
Sample  Nationality  Handedness
1   USA  Right-handed
2   Japan    Left-handed
3   USA  Right-handed
4   Japan    Right-handed
5   Japan    Left-handed
6   Japan    Right-handed
7   USA  Right-handed
8   USA  Left-handed
9   Japan    Right-handed
10  USA  Right-handed"""
data = pd.read_table(StringIO(data), sep=r'\s+')
data

### read : skiprows

In [ ]:
# pd.read_csv('data/ex4.csv')
pd.read_csv('data/ex4.csv', skiprows=[0, 2, 3])

In [ ]:
result = pd.read_csv('data/ex5.csv')
print(f'{result}\n\n'
      f'{pd.isnull(result)}')

### indicate na_values

In [ ]:
result1 = pd .read_csv('data/ex5.csv')
result2 = pd.read_csv('data/ex5.csv', na_values='foo')  # assign 'foo' = NaN
result1, '\n', result2

In [ ]:
sentinels = {'message': ['foo', 'NA'], 'something': ['two']}
pd.read_csv('data/ex5.csv', na_values=sentinels)

### Reading Text Files in Pieces

In [ ]:
result = pd.read_csv('data/ex6.csv')
result

### read : 5 rows

In [ ]:
pd.read_csv('data/ex6.csv', nrows=5)

### Writing Data to Text Format

In [ ]:
data = pd.read_csv('data/ex5.csv')
data, type(data)

In [ ]:
data.to_csv('data/out.csv')  # write to file CSV 

In [ ]:
import sys
data.to_csv(sys.stdout, sep='|')  # sys.stdout -> Screen,,  like print

### 爬蟲寫入 CSV 檔

In [ ]:
import pandas as pd

url = 'https://www.stockq.org/'
df = pd.read_html(url)

print(df[12].head())

df_data = df[12]

df_data.to_csv('./data/exchange_rate.csv', index=False, header=False)

## JSON Data
| 特性 | **Python 字典 (dict)** | **JSON (JavaScript Object Notation)** |
|:---:|:---:| --- |
| **本質** | 程式語言中的**資料結構** | 純文字**資料交換格式（字串）** |
| **存在狀態** | 存在於執行中的程式記憶體裡 | 可以存在於文字檔（.json）、網路傳輸中 |
| **字串包覆** | Key 和 Value 可以用單引號 `'` 或雙引號 `"` | **嚴格要求 Key 與字串 Value 必須使用雙引號 `"**` |

json.loads 用於解碼 JSON 資料。該函數返回 Python 欄位的資料 dict 類型。

In [ ]:
data = pd.read_json('data/example.json')
# df = pd.DataFrame.from_dict('data/example.json')
# df=pd.DataFrame.from_dict(result, orient="index")

data   # type(data)

### Convert the object to a JSON string.

In [ ]:
print(f'{data}\n\n'
      f'{data.to_json()}\n\n'
      f"{data.to_json(orient='records')}")

## save json

In [ ]:
data.to_json('./data/data.json', orient='records', indent=4)

In [ ]:
obj = ''' {"user_1": {"name": "Alice", "age": 25, "city": "Taipei"}, 
          "user_2": {"name": "Bob", "age": 30, "city": "Tainan"}}'''
type(obj)

### json load(str)

In [ ]:
import json
import pandas as pd
result = json.loads(obj)     # str                           # to dictionary
df=pd.DataFrame.from_dict(result, orient="index")
# df=pd.DataFrame.from_dict(result)
print(f'obj : {type(obj)}\n'
      f'\nobj ={obj}\n\n'                                 # string
      f'result :\t{type(result)}\nresult :{result}\n\n\n' #  dict
      f'type(df) :\t{type(df)}\n\ndf.T :\n{df.T}')
df

#### json.dumps 用於將 Python 在後端就需要將`字典`轉換成通用的 Json 格式。

In [ ]:
asjson = json.dumps(result)
print(f'{asjson},\n\n {type(asjson)}')

## 12-2 Reading Microsoft Excel Files & SQLite

In [ ]:
frame = pd.read_excel('data/ex1.xlsx', 'Sheet1')
frame

In [ ]:
frame.to_excel('data/ex2.xlsx', sheet_name='Sheet1')

### SQLite with Pandas
> #### 給你一個「工程師常用判斷公式」
>>是否會有`同時多人寫入`？
>> * 是 → MySQL / PostgreSQL<br>
>> * 否 → SQLite

面向               | SQLite           | MySQL / PostgreSQL
------------------:| ---------------- | ----------
架構               | 單一檔案          | Client–Server 
是否有 DB Server    | ❌ 沒有         | ✅ 有 
同時讀取            | 多人 OK          | 多人 OK
同時寫入            | `❌ 幾乎只能 1 個`| ✅ 多個
鎖定方式            | `整個資料庫鎖`    | Row / MVCC
適合資料量          | 小～中            | 中～超大
網路存取            | 不適合            | 原生支援
備份 / Replication | 很弱              | 非常成熟
權限管理            | 幾乎沒有          | 非常完整

In [ ]:
import pandas as pd
import sqlite3 as sql

df = pd.read_csv('./data/RegularSeasonCompactResults.csv')
df.head(), df.shape

In [ ]:
# connect to db. if db is not exist, it will create a new one
conn = sql.connect("./data/db_example.db")

In [ ]:
# insert dataframe into table. If table exist, it would replace it
# df.to_sql("example_table", conn, if_exists="replace", index=False)
df.to_sql("example_table", conn, if_exists="replace", index=False)

In [ ]:
# query from DB 
df_query = pd.read_sql_query("select * from example_table;", conn)
print(df_query.head(), '\n', df_query.shape)

In [ ]:
# insert dataframe into table. If table exist, it would replace it
df.to_sql("example_table", conn, if_exists="append", index=False)

In [ ]:
df_query = pd.read_sql_query("select * from example_table;", conn)
print(df_query.shape)

### Interacting with Databases

### drop existed table first

In [ ]:
import sqlite3
query = """drop table test;"""
con = sqlite3.connect('./data/mydata.sqlite')
con.execute(query)
con.commit()

### create a table

In [ ]:
import sqlite3
query = """
CREATE TABLE test
(a VARCHAR(20), b VARCHAR(20),
 c REAL,        d INTEGER
);"""
con = sqlite3.connect('./data/mydata.sqlite')
con.execute(query)
con.commit()

### empty db

In [ ]:
cursor = con.execute('select * from test')
rows = cursor.fetchall()
rows

In [ ]:
data = [('Atlanta', 'Georgia', 1.25, 6),
        ('Tallahassee', 'Florida', 2.6, 3),
        ('Sacramento', 'California', 1.7, 5)]
stmt = "INSERT INTO test VALUES(?, ?, ?, ?)"
con.executemany(stmt, data)
con.commit()

In [ ]:
cursor = con.execute('select * from test')
rows = cursor.fetchall()
rows

In [ ]:
# import pandas as pd
# cursor.description
pd.DataFrame(rows, columns=[x[0] for x in cursor.description])

---
# Mod 13 連接、合併和重塑 Data Wrangling : Join, Combine
> * 能夠找到、檢索、清理、處理、分析和表示不同類型的資料。<br>
> * 在很多應用中，資料通常散落在不同的檔或資料庫中，並不方便進行分析, 基本上不脫離 ETL (Extraction, Transform, Loading) 的範疇
>> * reshape：整形
>> * merge：歸併
>> * concatenate：串聯
>> * pivot：旋轉
>> * stack：堆疊

pandas 提供了多種便捷的工具，可以輕鬆地將 Series 和 DataFrame 物件組合在一起，並支援各種集合邏輯（用於索引）和關係代數功能（用於連接/合併類型的操作）。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Merging Datasets
pd.merge()：像 SQL 一樣的關聯查詢
當你有兩個 DataFrame，且它們之間有共同的欄位（例如 ID 或 姓名），你想把相關資訊「合併」在一起時使用。
* 方向：水平合併（增加欄位）。
* 關鍵點：依據內容（Key）來對齊。
* 常見參數：on (基準欄位), how ('inner', 'outer', 'left', 'right')。

### Database-Style DataFrame Joins
><img src="./img/merge.jpg"  style='width:100%'>

><img src="./img/merge01.jpg"  style='width:100%'>
><img src="./img/merge02.png"  style='width:100%'>

Pandas `merge()` 的 `how` 參數，直接對應了 SQL 的各種 `JOIN` 類型：

| 功能 | SQL 語法 | Pandas `merge()` 語法 | 說明 |
|:---:|:---:|:---:|:---:|
| 內連接 | `INNER JOIN` | `df1.merge(df2, how='inner')` | 只保留兩邊都有配對成功的資料（**預設值**） |
| 左外連接 | `LEFT JOIN` | `df1.merge(df2, how='left')` | 保留左邊所有資料，右邊沒配對到的補 `NaN` |
| 右外連接 | `RIGHT JOIN` | `df1.merge(df2, how='right')` | 保留右邊所有資料，左邊沒配對到的補 `NaN` |
| 全外連接 | `FULL OUTER JOIN` | `df1.merge(df2, how='outer')` | 保留兩邊所有的資料，沒配對到的地方補 `NaN` |
| 交叉連接 | `CROSS JOIN` | `df1.merge(df2, how='cross')` | 產生笛卡爾積（每個左邊列配對所有右邊列） |

In [ ]:
df1 = pd.DataFrame({'k1': ['b', 'b', 'a', 'c', 'a', 'a', 'b'], 'data1': range(7)})
df2 = pd.DataFrame({'k1': ['a', 'b', 'd'], 'data2': range(3)})
df1, df2

### 若未註明 on, 找到相同的 column 交集 (~ inner join)

In [ ]:
df1, df2, pd.merge(df1, df2)  # 找到相同的 column 交集 shift-tab "on"

In [ ]:
pd.merge(df1, df2, on='k1')  # column 'key' 相同

### 若未註明 on, 找到相同的 column 聯集 (~ outer / join) full

In [ ]:
df1, df2, pd.merge(df1, df2, how='outer') # 聯集 

In [ ]:
df3 = pd.DataFrame({'lkey': ['b', 'b', 'a', 'c', 'a', 'a', 'b'], 'data1': range(7)})
df4 = pd.DataFrame({'rkey': ['a', 'b','b', 'd'], 'data2': range(4)})
print(f'{df3}\n\n'
      f'{df4}\n\n'
      f"{pd.merge(df3, df4, how='outer', left_on='lkey', right_on='rkey')}")

### column 左右聯結 (~ left / right join)

In [ ]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'b'], 'data1': range(6)})
df2 = pd.DataFrame({'key': ['a', 'b', 'a', 'b', 'd'], 'data2': range(5)})
print(f'{df1}\n\n'
      f'{df2}\n\n'
      f"left merger :\n{pd.merge(df1, df2, on='key', how='left')}\n\n"
      f"right merger :\n{pd.merge(df1, df2, on='key', how='right')}")

### inner join = join

In [ ]:
df1, df2, pd.merge(df1, df2, how='inner'), pd.merge(df1, df2)

In [ ]:
left = pd.DataFrame({'key1': ['foo', 'foo', 'bar'],
                     'key2': ['one', 'two', 'one'],
                     'lval': [1, 2, 3]})
right = pd.DataFrame({'key1': ['foo', 'foo', 'bar', 'bar'],
                      'key2': ['one', 'one', 'one', 'two'],
                      'rval': [4, 5, 6, 7]})
print(f'{left}\n\n'
      f'{right}\n\n'
      f"{pd.merge(left, right, on=['key1', 'key2'], how='outer')}")

In [ ]:
print(f"{pd.merge(left, right, on='key1')}\n\n"
      f"{pd.merge(left, right, on='key1', suffixes=('_left', '_right'))}")

## Concatenating Along an Axis
> concat() 確實也有 join='inner' 和 join='outer' 參數。雖然 merge() 和 concat() 都有這兩個詞，但它們對齊的對象完全不同

> 關鍵點：`依據軸（Index 或 Column 名稱）來拼接`，`不考慮欄位內容的邏輯關聯`
> * pd.concat()：它的 Key 就是 Index（索引）`預設` 或 Column Name（欄位名）。它是根據「標籤名稱」在對齊。
> * pd.merge()：它的 Key 是 Data Values（資料內容）。它是根據「格子裡的數值」在對齊。

### Series

In [ ]:
ser1 = pd.Series(['A', 'B', 'C'], index=[1, 2, 3])
ser1

In [ ]:
ser2 = pd.Series(['D', 'E', 'F'], index=[4, 5, 6])
ser2

In [ ]:
pd.concat([ser1, ser2], axis=0)

In [ ]:
pd.concat([ser1, ser2], axis=1)

### DataFrame
預設情況下 `DataFrame` (i.e., axis=0)

In [ ]:
df1 = pd.DataFrame([['A1','B1','C1'], ['A2','B2','C2'], ['A3','B3','C3']], 
                   columns=['A', 'B', 'C'], 
                   index=[1, 2, 3])
df1

In [ ]:
df2 = pd.DataFrame([['B2','C2','D2'], ['B3','C3','D3'], ['B4','C4','D4']], 
                   columns=['B', 'C', 'D'], 
                   index=[2, 3, 4])
df2

In [ ]:
pd.concat([df1, df2])    # axis = 0, outer

In [ ]:
pd.concat([df1, df2], axis=1), pd.concat([df1, df2], axis='columns')   # axis=1

In [ ]:
pd.concat([df1, df2], ignore_index=True, join='outer', axis=0)    # ignore_index

In [ ]:
pd.concat([df1, df2], ignore_index=False, join='inner', axis=0)

In [ ]:
pd.concat([df1, df2], ignore_index=False, join='inner', axis=1)

In [ ]:
pd.concat([df1, df2], ignore_index=False, join='outer', axis=1)

## pandas vs. sql
><img src="./img/pd_vs_sql01.jpg"  style='width:80%'>

# `練習 Q02 (9,10)`

---
<center><h1>--- The End ---</h1><center>